<a href="https://colab.research.google.com/github/AliaF123/Laptop_Review/blob/main/laptop_review_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import plotly.express as px
import plotly.graph_objects as go
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
df = pd.read_excel('laptops_reviews.xlsx')

In [6]:
#preview data
df.head(10)

,product_name,overall_rating,no_ratings,no_reviews,rating,title,review,market,review_absa
0,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...,4.7,"15,210",900,5,Perfect product!,"Loved it, it's my first MacBook that I earned ...",Europe,The device is energy efficient and well-suited...
1,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...,4.7,"15,210",900,5,Fabulous!,Battery lasted longer than my first relationsh...,Europe,The device is energy efficient and well-suited...
2,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...,4.7,"15,210",900,5,Fabulous!,Such a great deal.. very happy with the perfor...,Europe,The device is energy efficient and well-suited...
3,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...,4.7,"15,210",900,4,Delightful,"Awesome build quality and very good display, b...",Europe,The device is energy efficient and well-suited...
4,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...,4.7,"15,210",900,5,Awesome,When i ordered and came to know about seller r...,Europe,The device is energy efficient and well-suited...
5,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...,4.7,"15,210",900,5,Super!,Super product,US,The laptop delivers strong performance for mul...
6,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...,4.7,"15,210",900,5,Super!,Go for it..its awesome,Europe,The device is energy efficient and well-suited...
7,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...,4.7,"15,210",900,5,Mind-blowing purchase,"Best , best and best 🫶🏻👑🍎",US,The laptop delivers strong performance for mul...
8,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...,4.7,"15,210",900,5,Just wow!,Its really very good and compact device.,Europe,The device is energy efficient and well-suited...
9,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...,4.7,"15,210",900,5,Brilliant,"Superb built quality, Amazing performance and ...",US,The laptop delivers strong performance for mul...


In [7]:
df['product_name']

,product_name
0,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...
1,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...
2,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...
3,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...
4,Apple MacBook AIR Apple M2 - (8 GB/256 GB SSD/...
...,...
24108,MSI Modern 14 Intel Core i5 13th Gen 1335U - (...
24109,MSI Modern 14 Intel Core i5 13th Gen 1335U - (...
24110,MSI Modern 14 Intel Core i5 13th Gen 1335U - (...
24111,Lenovo IdeaPad 5 2-in-1 WUXGA IPS AMD Ryzen 7 ...


In [8]:
print("Rows, Columns:", df.shape)

Rows, Columns: (24113, 9)


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24113 entries, 0 to 24112
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_name    24113 non-null  object 
 1   overall_rating  24113 non-null  float64
 2   no_ratings      24113 non-null  object 
 3   no_reviews      24113 non-null  object 
 4   rating          24113 non-null  int64  
 5   title           24113 non-null  object 
 6   review          24113 non-null  object 
 7   market          24113 non-null  object 
 8   review_absa     24113 non-null  object 
dtypes: float64(1), int64(1), object(7)
memory usage: 1.7+ MB


In [10]:
df.isnull().sum()

,0
product_name,0
overall_rating,0
no_ratings,0
no_reviews,0
rating,0
title,0
review,0
market,0
review_absa,0


In [11]:
df.duplicated().sum()

np.int64(3877)

In [27]:
def clean_review(text):

    # Handle missing values
    if pd.isna(text):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize
    words = text.split()

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [28]:
df['review'] = df['review'].astype(str)
df['clean_review'] = df['review'].apply(clean_review)

AttributeError: 'int' object has no attribute 'lower'

In [ ]:
df['clean_review']

In [ ]:

# Select the column that has reviews
text_column = "clean_review"

# Combine all reviews into one big text
text = " ".join(df[text_column].dropna().astype(str))

# Create word cloud
wc = WordCloud(
    width=900,
    height=450,
    background_color="white",
    stopwords={"the","is","and","a","to","it","this","was","for"}
).generate(text)

# Show result
plt.figure(figsize=(12,6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.show()